In [3]:
pip install timm


  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.8 MB/s  0:00:00
Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (485 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [timm]6/7 [timm]ngface_hub]
Note: you may need to restart the kernel to use updated packages.


In [4]:
# ==============================================================================
# PART 1: HYBRID FEATURE EXTRACTION (GPU REQUIRED)
# Switching to stable LeViT-256 to fix positional embedding mismatch
# ==============================================================================

import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import numpy as np
from tqdm import tqdm

# --- Configuration (UPDATED) ---
DATA_ROOT = "/workspace/" 
TRAIN_PATH = os.path.join(DATA_ROOT, "Train")
VALID_PATH = os.path.join(DATA_ROOT, "Valid")
TEST_PATH = os.path.join(DATA_ROOT, "Test")

BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# *** CRITICAL CHANGE: SWITCH TO A STABLE IMAGE SIZE ***
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224 # <--- Changed from 384 to 256 for stable LeViT-256 weights 

# --- Data Transformation ---
necessary_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Now 256x256
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# --- 1. Load Datasets Directly from Separate Folders ---
print(f"Loading data from: {DATA_ROOT}")
train_set = datasets.ImageFolder(TRAIN_PATH, transform=necessary_transform)
val_set = datasets.ImageFolder(VALID_PATH, transform=necessary_transform)
test_set = datasets.ImageFolder(TEST_PATH, transform=necessary_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Train samples: {len(train_set)}, Valid samples: {len(val_set)}, Test samples: {len(test_set)}")


# --- 2. Hybrid Feature Extractor Model (ViT-Base Update) ---
class HybridFeatureExtractor(nn.Module):
    """Fuses features from ViT-Base and EfficientNetV2-S."""
    def __init__(self):
        super().__init__()
        
        # 1. ViT-Base branch (Vision Transformer) - NEW MODEL
        self.vit = timm.create_model(
            'swin_small_patch4_window7_224', # <--- TARGET MODEL
            pretrained=True, 
            num_classes=0, 
            img_size=IMAGE_SIZE 
        )
        
        # 2. EfficientNetV2-S branch (CNN) - REMAINS THE SAME
        self.efficientnet = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)

    def forward(self, x):
        vit_features = self.vit(x) # Use self.vit
        efficient_features = self.efficientnet(x)
        # Feature Fusion
        combined_features = torch.cat((vit_features, efficient_features), dim=1)
        return combined_features

# --- 3. Feature Extraction Function ---
def extract_features(data_loader, split_name):
    """Extracts features and records time."""
    start_time = time.time()
    model.eval()
    all_features = []
    all_labels = []
    
    print(f"Starting feature extraction for {split_name}...")
    with torch.no_grad():
        for images, labels in tqdm(data_loader):
            images = images.to(DEVICE)
            features = model(images).cpu().numpy()
            all_features.append(features)
            all_labels.extend(labels.tolist())

    features_matrix = np.concatenate(all_features, axis=0)
    labels_array = np.array(all_labels)
    elapsed_time = time.time() - start_time
    
    return features_matrix, labels_array, elapsed_time

# Instantiate Model
model = HybridFeatureExtractor().to(DEVICE)

# Run Extraction for all splits
X_train, y_train, train_time_feat = extract_features(train_loader, "Training")
X_val, y_val, val_time_feat = extract_features(val_loader, "Validation")
X_test, y_test, test_time_feat = extract_features(test_loader, "Testing")

print(f"\nFeature Extraction Times:")
print(f"Train Feat Time: {train_time_feat:.2f}s | Val Feat Time: {val_time_feat:.2f}s | Test Feat Time: {test_time_feat:.2f}s")

# Save Extracted Features
np.save(os.path.join(DATA_ROOT, 'X_train.npy'), X_train)
np.save(os.path.join(DATA_ROOT, 'y_train.npy'), y_train)
np.save(os.path.join(DATA_ROOT, 'X_valid.npy'), X_val)
np.save(os.path.join(DATA_ROOT, 'y_valid.npy'), y_val)
np.save(os.path.join(DATA_ROOT, 'X_test.npy'), X_test)
np.save(os.path.join(DATA_ROOT, 'y_test.npy'), y_test)

Using device: cuda
Loading data from: /workspace/
Train samples: 18898, Valid samples: 2362, Test samples: 2364


model.safetensors:   0%|          | 0.00/200M [00:00<?, ?B/s]

Starting feature extraction for Training...


  0%|          | 0/296 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  2%|▏         | 6/296 [00:01<00:52,  5.52it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  4%|▍         | 12/296 [00:02<00:45,  6.19it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 10%|█         | 31/296 [00:08<00:52,  5.08it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 296/296 [01:12<00:00,  4.10it/s]


Starting feature extraction for Validation...


  0%|          | 0/37 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 11%|█         | 4/37 [00:01<00:08,  4.08it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 51%|█████▏    | 19/37 [00:06<00:09,  1.83it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 37/37 [00:10<00:00,  3.56it/s]


Starting feature extraction for Testing...


  0%|          | 0/37 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 16%|█▌        | 6/37 [00:01<00:05,  5.44it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 43%|████▎     | 16/37 [00:04<00:03,  5.38it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 37/37 [00:08<00:00,  4.21it/s]



Feature Extraction Times:
Train Feat Time: 72.39s | Val Feat Time: 10.41s | Test Feat Time: 8.82s


In [2]:
import optuna
import xgboost as xgb
import numpy as np
import time
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, log_loss, 
    roc_auc_score, confusion_matrix, classification_report
)

# --- CONFIGURATION ---
DATA_ROOT = "/workspace/"
N_TRIALS = 100 
SEED = 42

# --- 1. Utility Functions: Data Loading & Scaling ---
def load_and_scale_features(data_root):
    """Loads features from .npy files and applies StandardScaler."""
    
    X_train = np.load(os.path.join(data_root, 'X_train.npy'))
    y_train = np.load(os.path.join(data_root, 'y_train.npy'))
    X_val = np.load(os.path.join(data_root, 'X_valid.npy'))
    y_val = np.load(os.path.join(data_root, 'y_valid.npy'))
    X_test = np.load(os.path.join(data_root, 'X_test.npy'))
    y_test = np.load(os.path.join(data_root, 'y_test.npy'))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test) 

    # Convert labels to integer codes
    all_labels = pd.Series(np.concatenate([y_train, y_val, y_test]))
    label_map = {label: code for code, label in enumerate(all_labels.astype('category').cat.categories)}
    
    y_train_int = np.array([label_map[label] for label in y_train])
    y_val_int = np.array([label_map[label] for label in y_val])
    y_test_int = np.array([label_map[label] for label in y_test])
    
    num_classes = len(label_map)

    return X_train_scaled, y_train_int, X_val_scaled, y_val_int, X_test_scaled, y_test_int, num_classes, scaler


# --- 2. Utility Functions: Metric Calculation ---
def calculate_metrics(y_true, y_pred, y_prob, phase=""):
    """Calculates all required classification metrics."""
    metrics = {}
    
    # Standard Metrics (Macro Average)
    metrics[f'{phase} Accuracy'] = accuracy_score(y_true, y_pred)
    metrics[f'{phase} Precision'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics[f'{phase} Recall'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    metrics[f'{phase} F1 Score'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Loss and AUROC (Requires probabilities)
    metrics[f'{phase} Loss'] = log_loss(y_true, y_prob)
    metrics[f'{phase} AUROC'] = roc_auc_score(y_true, y_prob, multi_class='ovo', average='macro')

    # Calculate TPR and FPR (True/False Positive Rate) - Macro Averaged
    cm = confusion_matrix(y_true, y_pred)
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)
    
    epsilon = 1e-7
    TPR_macro = np.mean(TP / (TP + FN + epsilon)) 
    FPR_macro = np.mean(FP / (FP + TN + epsilon))

    metrics[f'{phase} TPR'] = TPR_macro
    metrics[f'{phase} FPR'] = FPR_macro
    
    return metrics


# --- 3. Optuna Objective Function (using xgb.train) ---
def objective(trial, X_train, y_train, X_val, y_val, num_classes):
    """XGBoost HPO objective function for Optuna using core API."""
    
    # 1. Define DMatrices for core API
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    # 2. Define the search space
    param = {
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'tree_method': 'hist', # Use 'exact' method with GPU
        'device': 'cuda',       # Explicitly set device to CUDA
        'num_class': num_classes,
        'verbosity': 0,                        
        'n_jobs': -1,                          
        'seed': SEED,
        
        # TPE Search Parameters
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
    }
    
    n_estimators = param.pop('n_estimators')

    # 3. Train using core API (xgb.train)
    model = xgb.train(
        param, 
        dtrain, 
        num_boost_round=n_estimators, 
        evals=[(dval, 'validation')], 
        early_stopping_rounds=30, 
        verbose_eval=False
    )
    
    # 4. Predict probabilities and find best iteration
    y_prob_val = model.predict(dval, iteration_range=(0, model.best_iteration))
    y_pred_val = np.argmax(y_prob_val, axis=1) 
    
    # Optimization target: Validation F1 Score (Macro)
    f1 = f1_score(y_val, y_pred_val, average='macro', zero_division=0)
    
    return f1

# --- 4. Main Execution (HPO and Final Evaluation) ---
def main():
    print(f"--- SWIN-SMALL HPO INITIATED ---")

    # Load and scale data
    X_train, y_train, X_val, y_val, X_test, y_test, num_classes, scaler = load_and_scale_features(DATA_ROOT)
    print(f"Data Loaded: {X_train.shape[0]} training samples, {num_classes} classes.")

    # --- HPO Run ---
    start_time_hpo = time.time()
    study = optuna.create_study(
        direction='maximize', 
        sampler=optuna.samplers.TPESampler(seed=SEED), 
        study_name='swin_small_hpo_xgb'
    )

    # ADDED PROGRESS BAR HERE
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, X_val, y_val, num_classes), 
        n_trials=N_TRIALS,
        show_progress_bar=True 
    )
    # END PROGRESS BAR

    time_hpo = time.time() - start_time_hpo
    print(f"Optuna HPO complete in {time_hpo:.2f} seconds.")

    best_params_raw = study.best_params
    best_n_estimators = best_params_raw.pop('n_estimators')
    
    best_params = {
            **best_params_raw,
            'objective': 'multi:softprob',
            'eval_metric': 'mlogloss',
            'tree_method': 'hist', # Use 'exact' method with GPU
            'device': 'cuda',       # Explicitly set device to CUDA
            'num_class': num_classes,
            'verbosity': 0,
            'seed': SEED
        }
    
    print("\nBest Hyperparameters Found (Optuna):")
    for key, value in best_params_raw.items():
        print(f"  {key}: {value}")
    print(f"  n_estimators (Best): {best_n_estimators}")

    # --- Final Model Training (Step 3) ---
    
    # Combine Train and Validation data for final training
    X_final_train = np.concatenate([X_train, X_val])
    y_final_train = np.concatenate([y_train, y_val])
    
    # Create final DMatrix
    dfinal_train = xgb.DMatrix(X_final_train, label=y_final_train)

    # Train Final Model (Measure training time)
    start_time_final_train = time.time()
    final_model = xgb.train(
        best_params,
        dfinal_train,
        num_boost_round=best_n_estimators
    )
    time_final_train = time.time() - start_time_final_train
    print(f"Final Model trained on (Train + Val) in {time_final_train:.4f} seconds.")


    # --- METRIC CALCULATION ---
    all_metrics = {}
    
    # Feature extraction times from your Part 1 run
    # Train Feat Time: 72.39s | Val Feat Time: 10.41s | Test Feat Time: 8.82s
    train_time_feat = 72.39
    val_time_feat = 10.41
    test_time_feat = 8.82
    
    # Create DMatrices for prediction
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # 1. Training Metrics
    start_time_train_eval = time.time()
    y_train_prob = final_model.predict(dtrain)
    y_train_pred = np.argmax(y_train_prob, axis=1)
    
    all_metrics.update(calculate_metrics(y_train, y_train_pred, y_train_prob, "Training"))
    all_metrics['Training Time (s)'] = time_final_train + train_time_feat 
    
    # 2. Validation Metrics
    start_time_val_eval = time.time()
    y_val_prob = final_model.predict(dval)
    y_val_pred = np.argmax(y_val_prob, axis=1)
    
    all_metrics.update(calculate_metrics(y_val, y_val_pred, y_val_prob, "Validation"))
    all_metrics['Validation Time (s)'] = (time.time() - start_time_val_eval) + val_time_feat 
    
    # 3. Test Metrics
    start_time_test_eval = time.time()
    y_test_prob = final_model.predict(dtest)
    y_test_pred = np.argmax(y_test_prob, axis=1)
    
    all_metrics.update(calculate_metrics(y_test, y_test_pred, y_test_prob, "Testing"))
    all_metrics['Testing Time (s)'] = (time.time() - start_time_test_eval) + test_time_feat 

    
    # --- FINAL OUTPUT ---
    print("\n" + "="*80)
    print("COMPLETE HPO PERFORMANCE REPORT (Swin-Small Hybrid + XGBoost)")
    print("="*80)
    
    # Define the exact order of metrics for clear reporting
    metric_order = [
        'Training Accuracy', 'Training Precision', 'Training Recall', 'Training F1 Score', 'Training Loss', 'Training TPR', 'Training FPR', 'Training Time (s)',
        'Validation Accuracy', 'Validation Precision', 'Validation Recall', 'Validation F1 Score', 'Validation Loss', 'Validation TPR', 'Validation FPR', 'Validation Time (s)',
        'Testing Accuracy', 'Testing Precision', 'Testing Recall', 'Testing F1 Score', 'Testing Loss', 'Testing TPR', 'Testing FPR', 'Testing AUROC', 'Testing Time (s)'
    ]

    for metric in metric_order:
        if metric in all_metrics:
            value = all_metrics[metric]
            if 'Time' in metric:
                print(f"{metric:<30} {value:.4f}")
            else:
                print(f"{metric:<30} {value:.6f}")

    print("\n" + "-"*30 + " TESTING CLASSIFICATION REPORT " + "-"*30)
    print(classification_report(y_test, y_test_pred, zero_division=0))

if __name__ == '__main__':
    main()

--- SWIN-SMALL HPO INITIATED ---


[I 2025-10-27 18:37:29,345] A new study created in memory with name: swin_small_hpo_xgb


Data Loaded: 18898 training samples, 9 classes.


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-27 18:37:53,616] Trial 0 finished with value: 0.9461198219453772 and parameters: {'n_estimators': 400, 'max_depth': 15, 'learning_rate': 0.08960785365368121, 'reg_alpha': 0.0024430162614261413, 'reg_lambda': 2.5361081166471375e-07, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'min_child_weight': 13}. Best is trial 0 with value: 0.9461198219453772.
[I 2025-10-27 18:40:02,840] Trial 1 finished with value: 0.9358071398613892 and parameters: {'n_estimators': 700, 'max_depth': 12, 'learning_rate': 0.010636066512540286, 'reg_alpha': 5.360294728728285, 'reg_lambda': 0.31044435499483225, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'min_child_weight': 3}. Best is trial 0 with value: 0.9461198219453772.
[I 2025-10-27 18:40:53,892] Trial 2 finished with value: 0.9414813705582047 and parameters: {'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.03647316284911211, 'reg_alpha': 4.17890272377219e-06, 'reg_lambda': 0.0032112643